In [2]:
import torch
pt_version = torch.__version__
print(f"Installing all PyG dependencies for PyTorch {pt_version}...")

# This installs scatter, sparse, cluster, and spline-conv
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{pt_version}.html



Installing all PyG dependencies for PyTorch 2.10.0+cu128...
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 40.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 36.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 44.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.1 MB/s eta 0:00:00


In [3]:
!pip install pygod==0.3.1 bioinfokit torch_geometric torchdata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 85.9 MB/s eta 0:00:00
  Created wheel for pygod: filename=pygod-0.3.1-py3-none-any.whl size=76463 sha256=dcdef305330b0cb018496172079cc41aa5e9e16c3c0812142cb94a03a3102dd9
  Stored in directory: /root/.cache/pip/wheels/64/f1/51/f483b59bd4d6893d4d556c9bf5698b45f0264bdcbf82e1cf0e
  Created wheel for bioinfokit: filename=bioinfokit-2.1.4-py3-none-any.whl size=59220 sha256=649de0c34c773b19a0e3c9e85151b186b86b4842d174a71f54b1d294bdb3d6be
  Stored in directory: /root/.cache/pip/wheels/b4/76/43/7fa2c349dac62f041fe8d85c9f48e47ca25fc39fd79d0b5f5e
Successfully built pygod bioinfokit


# Import Libraries

In [4]:
import sys
sys.path.append("..")
import seaborn as sb
# import dgl
import torch

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from tqdm import tqdm
import torch.nn.functional as F

import statistics
import argparse
import random

# from dgl.data import CitationGraphDataset
# from dgl.nn import GINConv, GraphConv, SAGEConv
import seaborn as sb
import torch
import torch.nn as nn
from torch.autograd import Variable
from scipy.optimize import linear_sum_assignment
import scipy
import scipy.optimize
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import sklearn as sk
import networkx as nx


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import random
import math
import time


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data

from torch_geometric.nn import GCNConv,GINConv,SAGEConv,GATConv,PNAConv, GraphSAGE
from torch_geometric.utils import add_self_loops
from torch_geometric.transforms import normalize_features
from pygod.utils import load_data
from pygod.utils.utility import check_parameter
from pygod.metrics import eval_roc_auc
from pygod.generator import gen_contextual_outliers, gen_structural_outliers
from scipy.linalg import sqrtm

# Utils

In [5]:

def _normalize(x):
    x_min = x.min()
    x_max = x.max()
    x_norm = (x - x_min)/x_max
    return x_norm

def gen_joint_structural_outliers(data, m, n, random_state=None):
    """
    We randomly select n nodes from the network which will be the anomalies
    and for each node we select m nodes from the network.
    We connect each of n nodes with the m other nodes.

    Parameters
    ----------
    data : PyTorch Geometric Data instance (torch_geometric.data.Data)
        The input data.
    m : int
        Number nodes in the outlier cliques.
    n : int
        Number of outlier cliques.
    p : int, optional
        Probability of edge drop in cliques. Default: ``0``.
    random_state : int, optional
        The seed to control the randomness, Default: ``None``.

    Returns
    -------
    data : PyTorch Geometric Data instance (torch_geometric.data.Data)
        The structural outlier graph with injected edges.
    y_outlier : torch.Tensor
        The outlier label tensor where 1 represents outliers and 0 represents
        regular nodes.
    """

    if not isinstance(data, Data):
        raise TypeError("data should be torch_geometric.data.Data")

    if isinstance(m, int):
        check_parameter(m, low=0, high=data.num_nodes, param_name='m')
    else:
        raise ValueError("m should be int, got %s" % m)

    if isinstance(n, int):
        check_parameter(n, low=0, high=data.num_nodes, param_name='n')
    else:
        raise ValueError("n should be int, got %s" % n)

    check_parameter(m * n, low=0, high=data.num_nodes, param_name='m*n')

    if random_state:
        np.random.seed(random_state)


    outlier_idx = np.random.choice(data.num_nodes, size=n, replace=False)
    all_nodes = [i for i in range(data.num_nodes)]
    rem_nodes = []

    for node in all_nodes:
        if node is not outlier_idx:
            rem_nodes.append(node)



    new_edges = []

    # connect all m nodes in each clique
    for i in range(0, n):
        other_idx = np.random.choice(data.num_nodes, size=m, replace=False)
        for j in other_idx:
            new_edges.append(torch.tensor([[i, j]], dtype=torch.long))


    new_edges = torch.cat(new_edges)


    y_outlier = torch.zeros(data.x.shape[0], dtype=torch.long)
    y_outlier[outlier_idx] = 1

    data.edge_index = torch.cat([data.edge_index, new_edges.T], dim=1)

    return data, y_outlier


def KL_neighbor_loss(predictions, targets, mask_len):
    x1 = predictions.squeeze().cpu().detach()
    x2 = targets.squeeze().cpu().detach()

    mean_x1 = x1.mean(0)
    mean_x2 = x2.mean(0)

    nn = x1.shape[0]
    h_dim = x1.shape[1]

    cov_x1 = (x1-mean_x1).transpose(1,0).matmul(x1-mean_x1) / max((nn-1),1)
    cov_x2 = (x2-mean_x2).transpose(1,0).matmul(x2-mean_x2) / max((nn-1),1)

    eye = torch.eye(h_dim)
    cov_x1 = cov_x1 + eye
    cov_x2 = cov_x2 + eye

    KL_loss = 0.5 * (math.log(torch.det(cov_x1) / torch.det(cov_x2)) - h_dim  + torch.trace(torch.inverse(cov_x2).matmul(cov_x1))
            + (mean_x2 - mean_x1).reshape(1,-1).matmul(torch.inverse(cov_x2)).matmul(mean_x2 - mean_x1))
    KL_loss = KL_loss.to(device)
    return KL_loss

def W2_neighbor_loss(predictions, targets, mask_len):

    x1 = predictions.squeeze().cpu().detach()
    x2 = targets.squeeze().cpu().detach()

    mean_x1 = x1.mean(0)
    mean_x2 = x2.mean(0)

    nn = x1.shape[0]

    cov_x1 = (x1-mean_x1).transpose(1,0).matmul(x1-mean_x1) / (nn-1)
    cov_x2 = (x2-mean_x2).transpose(1,0).matmul(x2-mean_x2) / (nn-1)


    W2_loss = torch.square(mean_x1-mean_x2).sum() + torch.trace(cov_x1 + cov_x2
                     + 2 * sqrtm(sqrtm(cov_x1) @ (cov_x2.numpy()) @ (sqrtm(cov_x1))))

    return W2_loss



# Layers

In [6]:
class MLP(nn.Module):
    def __init__(self, num_layers, input_dim, hidden_dim, output_dim):


        super(MLP, self).__init__()

        self.linear_or_not = True  # default is linear model
        self.num_layers = num_layers

        if num_layers < 1:
            raise ValueError("number of layers should be positive!")
        elif num_layers == 1:
            # Linear model
            self.linear = nn.Linear(input_dim, output_dim)
        else:
            # Multi-layer model
            self.linear_or_not = False
            self.linears = torch.nn.ModuleList()
            self.batch_norms = torch.nn.ModuleList()

            self.linears.append(nn.Linear(input_dim, hidden_dim))
            for layer in range(num_layers - 2):
                self.linears.append(nn.Linear(hidden_dim, hidden_dim))
            self.linears.append(nn.Linear(hidden_dim, output_dim))

            for layer in range(num_layers - 1):
                self.batch_norms.append(nn.BatchNorm1d((hidden_dim)))

    def forward(self, x):
        if self.linear_or_not:
            # If linear model
            return self.linear(x)
        else:
            # If MLP
            h = x
            for layer in range(self.num_layers - 1):
                h = self.linears[layer](h)

                if len(h.shape) > 2:
                    h = torch.transpose(h, 0, 1)
                    h = torch.transpose(h, 1, 2)

                h = self.batch_norms[layer](h)

                if len(h.shape) > 2:
                    h = torch.transpose(h, 1, 2)
                    h = torch.transpose(h, 0, 1)

                h = F.relu(h)
                # h = F.relu(self.linears[layer](h))

            return self.linears[self.num_layers - 1](h)


class MLP_generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLP_generator, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.linear2 = nn.Linear(output_dim, output_dim)
        self.linear3 = nn.Linear(output_dim, output_dim)
        self.linear4 = nn.Linear(output_dim, output_dim)

    def forward(self, embedding):
        neighbor_embedding = F.relu(self.linear(embedding))
        neighbor_embedding = F.relu(self.linear2(neighbor_embedding))
        neighbor_embedding = F.relu(self.linear3(neighbor_embedding))
        neighbor_embedding = self.linear4(neighbor_embedding)
        return neighbor_embedding


class PairNorm(nn.Module):
    def __init__(self, mode='PN', scale=10):

        assert mode in ['None', 'PN', 'PN-SI', 'PN-SCS']
        super(PairNorm, self).__init__()
        self.mode = mode
        self.scale = scale

        # Scale can be set based on origina data, and also the current feature lengths.
        # We leave the experiments to future. A good pool we used for choosing scale:
        # [0.1, 1, 10, 50, 100]
    def forward(self, x):
        if self.mode == 'None':
            return x
        col_mean = x.mean(dim=0)
        if self.mode == 'PN':
            x = x - col_mean
            rownorm_mean = (1e-6 + x.pow(2).sum(dim=1).mean()).sqrt()
            x = self.scale * x / rownorm_mean
        if self.mode == 'PN-SI':
            x = x - col_mean
            rownorm_individual = (1e-6 + x.pow(2).sum(dim=1, keepdim=True)).sqrt()
            x = self.scale * x / rownorm_individual
        if self.mode == 'PN-SCS':
            rownorm_individual = (1e-6 + x.pow(2).sum(dim=1, keepdim=True)).sqrt()
            x = self.scale * x / rownorm_individual - col_mean
        return x


# FNN
class FNN(nn.Module):
    def __init__(self, in_features, hidden, out_features, layer_num):
        super(FNN, self).__init__()
        self.linear1 = MLP(layer_num, in_features, hidden, out_features)
        self.linear2 = nn.Linear(out_features, out_features)
    def forward(self, embedding):
        x = self.linear1(embedding)
        x = self.linear2(F.relu(x))
        x = F.relu(x)
        return x


# GAD-NR

In [30]:
# %%
# Training
def train(data, y, yc, ys, yj, ysj, lr, epoch, device, encoder, lambda_loss1, lambda_loss2, lambda_loss3, hidden_dim, sample_size=10,loss_step=20,
          real_loss=False,calculate_contextual=False,calculate_structural=False):
    '''
     Main training function
     INPUT:
     -----------------------
     data : torch geometric dataset object
     lr    :    learning rate
     epoch     :    number of training epoch
     device     :   CPU or GPU
     encoder    :    GCN or GIN or GraphSAGE
     lambda_loss    :   Trade-off between degree loss and neighborhood reconstruction loss
     hidden_dim     :   latent variable dimension
    '''

    in_nodes = data.edge_index[0,:]
    out_nodes = data.edge_index[1,:]


    neighbor_dict = {}
    for in_node, out_node in zip(in_nodes, out_nodes):
        if in_node.item() not in neighbor_dict:
            neighbor_dict[in_node.item()] = []
        neighbor_dict[in_node.item()].append(out_node.item())

    neighbor_num_list = []
    for i in neighbor_dict:
        neighbor_num_list.append(len(neighbor_dict[i]))

    neighbor_num_list = torch.tensor(neighbor_num_list).to(device)

    in_dim = data.x.shape[1]
    GNNModel = GNNStructEncoder(in_dim, hidden_dim, hidden_dim, 2, sample_size, device=device,
                    neighbor_num_list=neighbor_num_list, GNN_name=encoder,
                    lambda_loss1=lambda_loss1, lambda_loss2=lambda_loss2,lambda_loss3=lambda_loss3)
    GNNModel.to(device)
    degree_params = list(map(id, GNNModel.degree_decoder.parameters()))
    base_params = filter(lambda p: id(p) not in degree_params,
                         GNNModel.parameters())

    opt = torch.optim.Adam([{'params': base_params}, {'params': GNNModel.degree_decoder.parameters(), 'lr': 1e-2}],lr=lr, weight_decay=0.0003)
    min_loss = float('inf')
    arg_min_loss_per_node = None

    best_auc = 0
    best_auc_contextual = 0
    best_auc_dense_structural = 0
    best_auc_joint_structural = 0
    best_auc_structure_type = 0


    loss_values = []
    for i in tqdm(range(1,epoch+1,1)):

        if i%loss_step==0:
            GNNModel.lambda_loss2 = GNNModel.lambda_loss2 + 0.5
            GNNModel.lambda_loss3 = GNNModel.lambda_loss3 / 2

        loss,loss_per_node,h_loss,degree_loss,feature_loss = GNNModel(data.edge_index, data.x, neighbor_num_list, neighbor_dict, device=device)

        loss_per_node = loss_per_node.cpu().detach()

        h_loss = h_loss.cpu().detach()
        degree_loss = degree_loss.cpu().detach()
        feature_loss = feature_loss.cpu().detach()

        # Add epsilon (1e-8) to prevent division by zero
        epsilon = 1e-8
        h_loss_norm = h_loss / (torch.max(h_loss) - torch.min(h_loss) + epsilon)
        degree_loss_norm = degree_loss / (torch.max(degree_loss) - torch.min(degree_loss) + epsilon)
        feature_loss_norm = feature_loss / (torch.max(feature_loss) - torch.min(feature_loss) + epsilon)

        comb_loss = args.h_loss_weight * h_loss_norm + args.degree_loss_weight * degree_loss_norm + args.feature_loss_weight * feature_loss_norm

        # Safety net to catch any rogue Infs/NaNs before sklearn sees them
        comb_loss = torch.nan_to_num(comb_loss)
        loss_per_node = torch.nan_to_num(loss_per_node)

        if real_loss:
            comp_loss = loss_per_node
        else:
            comp_loss = comb_loss

        auc_score = eval_roc_auc(y.numpy(), comp_loss.numpy()) * 100
        print("Dataset Name: ",dataset_str, ", AUC Score(benchmark/combined): ", auc_score)
        best_auc = max(best_auc, auc_score)

        # Only evaluate fake anomaly scores if they were actually injected
        if yc.sum() > 0:
            contextual_auc_score = eval_roc_auc(yc.numpy(), comp_loss.numpy()) * 100
            print("Dataset Name: ",dataset_str, ", AUC Score (contextual): ", contextual_auc_score)
            best_auc_contextual = max(best_auc_contextual, contextual_auc_score)

        if ys.sum() > 0:
            dense_structural_auc_score = eval_roc_auc(ys.numpy(), comp_loss.numpy()) * 100
            print("Dataset Name: ",dataset_str, ", AUC Score (structural): ", dense_structural_auc_score)
            best_auc_dense_structural = max(best_auc_dense_structural, dense_structural_auc_score)

        if yj.sum() > 0:
            joint_type_auc_score = eval_roc_auc(yj.numpy(), comp_loss.numpy()) * 100
            print("Dataset Name: ",dataset_str, ", AUC Score (joint-type): ", joint_type_auc_score)
            best_auc_joint_structural = max(best_auc_joint_structural, joint_type_auc_score)

        if ysj.sum() > 0:
            structure_type_auc_score = eval_roc_auc(ysj.numpy(), comp_loss.numpy()) * 100
            print("Dataset Name: ",dataset_str, ", AUC Score (structure type): ", structure_type_auc_score)
            best_auc_structure_type = max(best_auc_structure_type, structure_type_auc_score)

        print("===========================================================================================")
        print("Dataset Name: ",dataset_str, " Best AUC Score(benchmark/combined): ", best_auc)

        if yc.sum() > 0:
            print("Dataset Name: ",dataset_str, " Best AUC Score (contextual): ", best_auc_contextual)
        if ys.sum() > 0:
            print("Dataset Name: ",dataset_str, " Best AUC Score (structural): ", best_auc_dense_structural)
        if yj.sum() > 0:
            print("Dataset Name: ",dataset_str, " Best AUC Score (joint-type): ", best_auc_joint_structural)
        if ysj.sum() > 0:
            print("Dataset Name: ",dataset_str, " Best AUC Score (structure type): ", best_auc_structure_type)
        print("===========================================================================================")

        if loss < min_loss:
            min_loss = loss
            arg_min_loss_per_node = loss_per_node
        opt.zero_grad()
        loss.backward()
        opt.step()

        loss = loss.cpu().detach()
        loss_values.append(loss)

        en = time.time()

    # --- PLOTTING COMPLETELY DISABLED AS REQUESTED ---
    # if args.plot_loss:
    #     plt.plot(np.array(loss_values), 'r')
    #     plt.show()

    return min_loss.item(), arg_min_loss_per_node.cpu().detach()


def evaluate(model, embeddings, labels, mask):
    model.eval()
    with torch.no_grad():
        logits = model(embeddings)
        logits = logits[mask]
        labels = labels[mask]
        _, indices = torch.max(logits, dim=1)
        correct = torch.sum(indices == labels)
        return correct.item() * 1.0 / len(labels)


def train_real_datasets(dataset_str, epoch_num = 10, lr = 5e-6, encoder = "GCN",
                        lambda_loss1=1e-2, lambda_loss2=1e-3, lambda_loss3=1e-3, sample_size=8, loss_step=20, hidden_dim=None,
                        real_loss=False,calculate_contextual=False,calculate_structural=False):

    data = load_data(dataset_str)
    node_features = data.x

    if args.normalize_feat:
        node_features_min = node_features.min()
        node_features_max = node_features.max()
        node_features = (node_features - node_features_min)/node_features_max
        data.x = node_features

    # Initialize as zero tensors instead of empty lists
    num_nodes = data.x.shape[0]
    yc = torch.zeros(num_nodes, dtype=torch.long)
    ys = torch.zeros(num_nodes, dtype=torch.long)
    yj = torch.zeros(num_nodes, dtype=torch.long)

    if calculate_contextual:
        if dataset_str == "inj_cora":
            yc = data.y >> 0 & 1 # contextual outliers
        else:
            data, yc = gen_contextual_outliers(data=data,n=args.contextual_n,k=args.contextual_k)

        yc = yc.cpu().detach()

    if calculate_structural:
        if dataset_str == "inj_cora":
            ys = data.y >> 1 & 1 # structural outliers
        else:
            data, ys = gen_structural_outliers(data=data,n=args.structural_n,m=args.structural_m,p=0.2)

        ys = ys.cpu().detach()
        data, yj = gen_joint_structural_outliers(data=data,n=args.structural_n,m=args.structural_m)
        yj = yj.cpu().detach() # Ensure yj is detached

    if args.use_combine_outlier:
        data.y = torch.logical_or(ys, yc).int()

    # Safely compute 0 OR 0 if structural is False
    ysj = torch.logical_or(ys, yj).int()

    y = data.y.bool()    # binary labels (inlier/outlier)
    y = y.cpu().detach()

    edge_index = data.edge_index.cpu()

    self_edges = torch.tensor([[i for i in range(num_nodes)],[i for i in range(num_nodes)]])
    edge_index = torch.cat([edge_index,self_edges],dim=1)
    data.edge_index = edge_index
    data = data.to(device)

    loss, loss_per_node, = train(data, y, yc, ys, yj, ysj, lr=lr, epoch=epoch_num, device=device, encoder=encoder, lambda_loss1=lambda_loss1,
          lambda_loss2=lambda_loss2, lambda_loss3=lambda_loss3, hidden_dim=hidden_dim, sample_size=sample_size,loss_step=loss_step,
                                 real_loss=real_loss,calculate_contextual=calculate_contextual,calculate_structural=calculate_structural)

In [8]:
# generate ground truth neighbors Hv
def generate_gt_neighbor(neighbor_dict, node_embeddings, neighbor_num_list, in_dim):
    max_neighbor_num = max(neighbor_num_list)
    all_gt_neighbor_embeddings = []
    for i, embedding in enumerate(node_embeddings):
        neighbor_indexes = neighbor_dict[i]
        neighbor_embeddings = []
        for index in neighbor_indexes:
            neighbor_embeddings.append(node_embeddings[index].tolist())
        if len(neighbor_embeddings) < max_neighbor_num:
            for _ in range(max_neighbor_num - len(neighbor_embeddings)):
                neighbor_embeddings.append(torch.zeros(in_dim).tolist())
        all_gt_neighbor_embeddings.append(neighbor_embeddings)
    return all_gt_neighbor_embeddings


# Main Autoencoder structure here
class GNNStructEncoder(nn.Module):
    def __init__(self, in_dim0, in_dim, hidden_dim, layer_num, sample_size, device, neighbor_num_list,
                 GNN_name="GIN", norm_mode="PN-SCS", norm_scale=20, lambda_loss1=0.01, lambda_loss2=0.001, lambda_loss3=0.0001):

        super(GNNStructEncoder, self).__init__()

        self.mlp0 = nn.Linear(in_dim0, hidden_dim)
        self.norm = PairNorm(norm_mode, norm_scale)
        self.out_dim = hidden_dim
        self.lambda_loss1 = lambda_loss1
        self.lambda_loss2 = lambda_loss2
        self.lambda_loss3 = lambda_loss3
        # GNN Encoder
        if GNN_name == "GIN":
            self.linear1 = MLP(layer_num, hidden_dim, hidden_dim, hidden_dim)
            self.graphconv1 = GINConv(self.linear1)
            self.linear2 = MLP(layer_num, hidden_dim, hidden_dim, hidden_dim)
            self.graphconv2 = GINConv(self.linear2)
        elif GNN_name == "GCN":
            self.graphconv1 = GCNConv(hidden_dim, hidden_dim)
            self.graphconv2 = GCNConv(hidden_dim, hidden_dim)
        elif GNN_name == "GAT":
            self.graphconv1 = GATConv(hidden_dim, hidden_dim)
            self.graphconv2 = GATConv(hidden_dim, hidden_dim)
        else:
            self.graphconv1 = SAGEConv(hidden_dim, hidden_dim, aggr=args.aggregator)
            # self.graphconv2 = SAGEConv(hidden_dim, hidden_dim, aggr='mean')

            # self.graphconv1 = GraphSAGE(hidden_dim, hidden_dim, aggr='mean', num_layers=1)


        self.neighbor_num_list = neighbor_num_list
        self.tot_node = len(neighbor_num_list)

        self.gaussian_mean = nn.Parameter(
            torch.FloatTensor(sample_size, hidden_dim).uniform_(-0.5 / hidden_dim,
                                                                                     0.5 / hidden_dim)).to(device)
        self.gaussian_log_sigma = nn.Parameter(
            torch.FloatTensor(sample_size, hidden_dim).uniform_(-0.5 / hidden_dim,
                                                                                     0.5 / hidden_dim)).to(device)
        self.m = torch.distributions.Normal(torch.zeros(sample_size, hidden_dim),
                                            torch.ones(sample_size, hidden_dim))

        self.m_batched = torch.distributions.Normal(torch.zeros(sample_size, self.tot_node, hidden_dim),
                                            torch.ones(sample_size, self.tot_node, hidden_dim))

        self.m_h = torch.distributions.Normal(torch.zeros(sample_size, hidden_dim),
                                            50* torch.ones(sample_size, hidden_dim))

        # Before MLP Gaussian Means, and std

        self.mlp_gaussian_mean = nn.Parameter(
            torch.FloatTensor(hidden_dim).uniform_(-0.5 / hidden_dim, 0.5 / hidden_dim)).to(device)
        self.mlp_gaussian_log_sigma = nn.Parameter(
            torch.FloatTensor(hidden_dim).uniform_(-0.5 / hidden_dim, 0.5 / hidden_dim)).to(device)
        self.mlp_m = torch.distributions.Normal(torch.zeros(hidden_dim), torch.ones(hidden_dim))

        self.mlp_mean = FNN(hidden_dim, hidden_dim, hidden_dim, 3)
        self.mlp_sigma = FNN(hidden_dim, hidden_dim, hidden_dim, 3)
        self.softplus = nn.Softplus()

        self.mean_agg = SAGEConv(hidden_dim, hidden_dim, aggr=args.aggregator, normalize = False)
        # self.mean_agg = GraphSAGE(hidden_dim, hidden_dim, aggr='mean', num_layers=1)
        self.std_agg = PNAConv(hidden_dim, hidden_dim, aggregators=["std"],scalers=["identity"], deg=neighbor_num_list)
        self.layer1_generator = MLP_generator(hidden_dim, hidden_dim)

        # Decoders
        self.degree_decoder = FNN(hidden_dim, hidden_dim, 1, 4)
        self.feature_decoder = FNN(hidden_dim, hidden_dim, in_dim, 3)
        self.degree_loss_func = nn.MSELoss()
        self.feature_loss_func = nn.MSELoss()
        self.pool = mp.Pool(4)
        self.in_dim = in_dim
        self.sample_size = sample_size
        self.init_projection = FNN(in_dim, hidden_dim, hidden_dim, 1)


    def forward_encoder(self, x, edge_index):

        # Apply graph convolution and activation, pair-norm to avoid trivial solution
        h0 = self.mlp0(x)
        l1 = self.graphconv1(h0, edge_index)
        return l1, h0



    # Sample neighbors from neighbor set, if the length of neighbor set less than sample size, then do the padding.
    def sample_neighbors(self, indexes, neighbor_dict, gt_embeddings):
        sampled_embeddings_list = []
        mark_len_list = []
        for index in indexes:
            sampled_embeddings = []
            neighbor_indexes = neighbor_dict[index]
            if len(neighbor_indexes) < self.sample_size:
                mask_len = len(neighbor_indexes)
                sample_indexes = neighbor_indexes
            else:
                sample_indexes = random.sample(neighbor_indexes, self.sample_size)
                mask_len = self.sample_size
            for index in sample_indexes:
                sampled_embeddings.append(gt_embeddings[index].tolist())
            if len(sampled_embeddings) < self.sample_size:
                for _ in range(self.sample_size - len(sampled_embeddings)):
                    sampled_embeddings.append(torch.zeros(self.out_dim).tolist())
            sampled_embeddings_list.append(sampled_embeddings)
            mark_len_list.append(mask_len)

        return sampled_embeddings_list, mark_len_list

    def reconstruction_neighbors2(self, l1, h0, edge_index):

        recon_loss = 0
        recon_loss_per_node = []

        sample_sz_per_node = [self.sample_size]* self.tot_node

        mean_neigh = self.mean_agg.forward(h0, edge_index).detach()
        std_neigh = self.std_agg(h0, edge_index).detach()


        cov_neigh = torch.bmm(std_neigh.unsqueeze(dim=-1),std_neigh.unsqueeze(dim=1))

        target_mean = mean_neigh
        target_cov = cov_neigh

        self_embedding = l1
        # self_embedding = _normalize(self_embedding)

        self_embedding = self_embedding.unsqueeze(0)
        self_embedding = self_embedding.repeat(self.sample_size, 1, 1)
        generated_mean = self.mlp_mean(self_embedding)
        generated_sigma = self.mlp_sigma(self_embedding)


        std_z = self.m_batched.sample().to(device)
        var = generated_mean + generated_sigma.exp() * std_z
        nhij = self.layer1_generator(var)

        generated_mean = torch.mean(nhij,dim=0)
        generated_std = torch.std(nhij,dim=0)
        generated_cov = torch.bmm(generated_std.unsqueeze(dim=-1),generated_std.unsqueeze(dim=1))/self.sample_size


        tot_nodes = l1.shape[0]
        h_dim = l1.shape[1]

        single_eye = torch.eye(h_dim).to(device)
        single_eye = single_eye.unsqueeze(dim=0)
        batch_eye = single_eye.repeat(tot_nodes,1,1)

        target_cov = target_cov + batch_eye
        generated_cov = generated_cov + batch_eye


        det_target_cov = torch.linalg.det(target_cov)
        det_generated_cov = torch.linalg.det(generated_cov)
        trace_mat = torch.matmul(torch.inverse(generated_cov),target_cov)


        x = torch.bmm(torch.unsqueeze(generated_mean - target_mean,dim=1),torch.inverse(generated_cov))
        y = torch.unsqueeze(generated_mean - target_mean,dim=-1)
        z = torch.bmm(x,y).squeeze()

        KL_loss = 0.5 * (torch.log(det_target_cov / det_generated_cov) - h_dim  + trace_mat.diagonal(offset=0, dim1=-1, dim2=-2).sum(-1) + z)

        recon_loss = torch.mean(KL_loss)
        recon_loss_per_node = KL_loss


        return recon_loss, recon_loss_per_node


    def reconstruction_neighbors(self, FNN_generator, neighbor_indexes, neighbor_dict, from_layer, to_layer, device):


        local_index_loss = 0
        local_index_loss_per_node = []
        sampled_embeddings_list, mark_len_list = self.sample_neighbors(neighbor_indexes, neighbor_dict, to_layer)
        for i, neighbor_embeddings1 in enumerate(sampled_embeddings_list):
            # Generating h^k_v, reparameterization trick

            # print(len(neighbor_embeddings1))


            index = neighbor_indexes[i]
            mask_len1 = mark_len_list[i]
            mean = from_layer[index].repeat(self.sample_size, 1)
            mean = self.mlp_mean(mean)
            sigma = from_layer[index].repeat(self.sample_size, 1)
            sigma = self.mlp_sigma(sigma)
            std_z = self.m.sample().to(device)
            var = mean + sigma.exp() * std_z
            nhij = FNN_generator(var)

            generated_neighbors = nhij
            sum_neighbor_norm = 0

            for indexi, generated_neighbor in enumerate(generated_neighbors):
                sum_neighbor_norm += torch.norm(generated_neighbor) / math.sqrt(self.out_dim)
            generated_neighbors = torch.unsqueeze(generated_neighbors, dim=0).to(device)
            target_neighbors = torch.unsqueeze(torch.FloatTensor(neighbor_embeddings1), dim=0).to(device)

            if args.neigh_loss == "KL":
                KL_loss = KL_neighbor_loss(generated_neighbors, target_neighbors, mask_len1)
                local_index_loss += KL_loss
                local_index_loss_per_node.append(KL_loss)

            else:
                W2_loss = W2_neighbor_loss(generated_neighbors, target_neighbors, mask_len1)
                local_index_loss += W2_loss
                local_index_loss_per_node.append(W2_loss)


        local_index_loss_per_node = torch.stack(local_index_loss_per_node)
        return local_index_loss, local_index_loss_per_node


    def neighbor_decoder(self, gij, ground_truth_degree_matrix, h0, neighbor_dict, device, h, edge_index):

        # Degree decoder below:
        tot_nodes = gij.shape[0]
        degree_logits = self.degree_decoding(gij)
        ground_truth_degree_matrix = torch.unsqueeze(ground_truth_degree_matrix, dim=1)
        degree_loss = self.degree_loss_func(degree_logits, ground_truth_degree_matrix.float())
        degree_loss_per_node = (degree_logits-ground_truth_degree_matrix).pow(2)
        _, degree_masks = torch.max(degree_logits.data, dim=1)
        h_loss = 0
        feature_loss = 0
        # layer 1
        loss_list = []
        loss_list_per_node = []
        feature_loss_list = []
        # Sample multiple times to remove noise
        for _ in range(3):
            local_index_loss_sum = 0
            local_index_loss_sum_per_node = []
            indexes = []
            h0_prime = self.feature_decoder(gij)
            feature_losses = self.feature_loss_func(h0, h0_prime)
            feature_losses_per_node = (h0-h0_prime).pow(2).mean(1)
            feature_loss_list.append(feature_losses_per_node)



            local_index_loss, local_index_loss_per_node = self.reconstruction_neighbors2(gij,h0,edge_index)

            loss_list.append(local_index_loss)
            loss_list_per_node.append(local_index_loss_per_node)

        loss_list = torch.stack(loss_list)
        h_loss += torch.mean(loss_list)

        loss_list_per_node = torch.stack(loss_list_per_node)
        h_loss_per_node = torch.mean(loss_list_per_node,dim=0)

        feature_loss_per_node = torch.mean(torch.stack(feature_loss_list),dim=0)
        feature_loss += torch.mean(torch.stack(feature_loss_list))

        h_loss_per_node = h_loss_per_node.reshape(tot_nodes,1)
        degree_loss_per_node = degree_loss_per_node.reshape(tot_nodes,1)
        feature_loss_per_node = feature_loss_per_node.reshape(tot_nodes,1)

        loss = self.lambda_loss1 * h_loss + degree_loss * self.lambda_loss3 + self.lambda_loss2 * feature_loss
        loss_per_node = self.lambda_loss1 * h_loss_per_node + degree_loss_per_node * self.lambda_loss3 + self.lambda_loss2 * feature_loss_per_node


        return loss,loss_per_node,h_loss_per_node,degree_loss_per_node,feature_loss_per_node

    def degree_decoding(self, node_embeddings):
        degree_logits = F.relu(self.degree_decoder(node_embeddings))
        return degree_logits

    def forward(self, edge_index, x, ground_truth_degree_matrix, neighbor_dict, device):

        # Generate GNN encodings
        l1, h0 = self.forward_encoder(x, edge_index)
        loss, loss_per_node,h_loss,degree_loss,feature_loss = self.neighbor_decoder(l1, ground_truth_degree_matrix, h0, neighbor_dict, device, x, edge_index)

        return loss, loss_per_node,h_loss,degree_loss,feature_loss


# Execution

In [22]:
# --- 3. MAIN ARGUMENTS (Corrected to match paper baselines) ---
class Args:
    dataset = "books"
    lr = 0.01
    epoch_num = 500

    # --- STRICT BASELINE LAMBDAS ---
    lambda_loss1 = 0.001         # Neighbor (\lambda_n in paper)
    lambda_loss2 = 0.8           # Feature  (\lambda_x in paper)
    lambda_loss3 = 0.5           # Degree   (\lambda_d in paper)

    sample_size = 10
    dimension = 16               # Books uses dim 16
    encoder = "GCN"              # Paper uses GCN for baselines

    # --- TRAINING STABILIZATION ---
    loss_step = 1000             # Set > epoch_num to disable dynamic weight mutation
    real_loss = False            # Use normalized comb_loss for balanced scales

    neigh_loss = "KL"
    h_loss_weight = 1.0
    feature_loss_weight = 2.0
    degree_loss_weight = 1.0

    # --- NO DATA POISONING FOR TABLE 2 ---
    calculate_contextual = False # Use real ground-truth labels only
    contextual_n = 14
    contextual_k = 5
    calculate_structural = False # Use real ground-truth labels only
    structural_n = 14
    structural_m = 5

    use_combine_outlier = False  # False because Books has real ground-truth labels
    plot_loss = True
    normalize_feat = True
    aggregator = "mean"

args = Args()
dataset_str = args.dataset

In [31]:
# ==========================================================
# 1. MAIN RUN (Full GAD-NR Baseline)
# ==========================================================
print("\n=======================================")
print(f"        STARTING MAIN RUN (BOOKS)      ")
print("=======================================\n")
for run in range(1):
    train_real_datasets(
        dataset_str=args.dataset, lr=args.lr, epoch_num=args.epoch_num,
        lambda_loss1=args.lambda_loss1, lambda_loss2=args.lambda_loss2, lambda_loss3=args.lambda_loss3,
        encoder=args.encoder, sample_size=args.sample_size, loss_step=args.loss_step,
        hidden_dim=args.dimension, real_loss=args.real_loss,
        calculate_contextual=args.calculate_contextual, calculate_structural=args.calculate_structural
    )




        STARTING MAIN RUN (BOOKS)      

Loading books.pt...


  1%|          | 4/500 [00:00<00:27, 17.72it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  43.969681397738945
Dataset Name:  books  Best AUC Score(benchmark/combined):  43.969681397738945
Dataset Name:  books , AUC Score(benchmark/combined):  42.71582733812949
Dataset Name:  books  Best AUC Score(benchmark/combined):  43.969681397738945
Dataset Name:  books , AUC Score(benchmark/combined):  47.2160842754368
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.2160842754368
Dataset Name:  books , AUC Score(benchmark/combined):  47.50513874614594
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.50513874614594


  2%|▏         | 8/500 [00:00<00:26, 18.58it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  42.85714285714286
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.50513874614594
Dataset Name:  books , AUC Score(benchmark/combined):  46.37461459403905
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.50513874614594
Dataset Name:  books , AUC Score(benchmark/combined):  45.873586844809864
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.50513874614594
Dataset Name:  books , AUC Score(benchmark/combined):  46.721479958890036
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.50513874614594


...

 99%|█████████▊| 493/500 [00:27<00:00, 19.87it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  62.21608427543679
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  61.08941418293937
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  62.84686536485097
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063


100%|█████████▉| 498/500 [00:27<00:00, 20.43it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  55.5524152106886
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  62.70811921891058
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  61.13309352517986
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  69.17009249743063


100%|██████████| 500/500 [00:27<00:00, 18.15it/s]


In [32]:

# ==========================================================
# 2. ABLATION: NO FEATURE RECONSTRUCTION
# ==========================================================
print("\n=======================================")
print(f"    ABLATION 1: NO FEATURE RECONSTRUCTION  ")
print("=======================================\n")
for run in range(1):
    train_real_datasets(
        dataset_str=args.dataset, lr=args.lr, epoch_num=args.epoch_num,
        lambda_loss1=args.lambda_loss1, lambda_loss2=0.0, lambda_loss3=args.lambda_loss3, # feature=0
        encoder=args.encoder, sample_size=args.sample_size, loss_step=args.loss_step,
        hidden_dim=args.dimension, real_loss=args.real_loss,
        calculate_contextual=args.calculate_contextual, calculate_structural=args.calculate_structural
    )




    ABLATION 1: NO FEATURE RECONSTRUCTION  

Loading books.pt...


  0%|          | 2/500 [00:00<00:30, 16.15it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  43.517471736896205
Dataset Name:  books  Best AUC Score(benchmark/combined):  43.517471736896205
Dataset Name:  books , AUC Score(benchmark/combined):  46.22816032887975
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.22816032887975
Dataset Name:  books , AUC Score(benchmark/combined):  46.04573484069887
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.22816032887975
Dataset Name:  books , AUC Score(benchmark/combined):  45.413669064748206
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.22816032887975
Dataset Name:  books , AUC Score(benchmark/combined):  47.63617677286742
Dataset Name:  books  Best AUC Score(benchmark/combined):  47.63617677286742


  2%|▏         | 9/500 [00:00<00:26, 18.73it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  48.4134121274409
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.4134121274409
Dataset Name:  books , AUC Score(benchmark/combined):  46.28597122302159
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.4134121274409
Dataset Name:  books , AUC Score(benchmark/combined):  46.41572456320658
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.4134121274409
Dataset Name:  books , AUC Score(benchmark/combined):  47.10688591983556
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.4134121274409


...

Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  61.04573484069887
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961


 99%|█████████▉| 496/500 [00:27<00:00, 15.81it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961


100%|██████████| 500/500 [00:27<00:00, 17.96it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  61.11125385405961
Dataset Name:  books , AUC Score(benchmark/combined):  62.09403905447072
Dataset Name:  books  Best AUC Score(benchmark/combined):  62.09403905447072
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  62.09403905447072


In [33]:
# ==========================================================
# 3. ABLATION: NO DEGREE RECONSTRUCTION
# ==========================================================
print("\n=======================================")
print(f"    ABLATION 2: NO DEGREE RECONSTRUCTION   ")
print("=======================================\n")
for run in range(1):
    train_real_datasets(
        dataset_str=args.dataset, lr=args.lr, epoch_num=args.epoch_num,
        lambda_loss1=args.lambda_loss1, lambda_loss2=args.lambda_loss2, lambda_loss3=0.0, # degree=0
        encoder=args.encoder, sample_size=args.sample_size, loss_step=args.loss_step,
        hidden_dim=args.dimension, real_loss=args.real_loss,
        calculate_contextual=args.calculate_contextual, calculate_structural=args.calculate_structural
    )



    ABLATION 2: NO DEGREE RECONSTRUCTION   

Loading books.pt...


  0%|          | 2/500 [00:00<00:38, 13.04it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  44.285714285714285
Dataset Name:  books  Best AUC Score(benchmark/combined):  44.285714285714285
Dataset Name:  books , AUC Score(benchmark/combined):  43.24768756423432
Dataset Name:  books  Best AUC Score(benchmark/combined):  44.285714285714285
Dataset Name:  books , AUC Score(benchmark/combined):  46.644398766700924
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.644398766700924


  1%|          | 6/500 [00:00<00:33, 14.84it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  46.492805755395686
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.644398766700924
Dataset Name:  books , AUC Score(benchmark/combined):  48.35303186022611
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.35303186022611
Dataset Name:  books , AUC Score(benchmark/combined):  43.86176772867421
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.35303186022611
Dataset Name:  books , AUC Score(benchmark/combined):  47.335560123329905
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.35303186022611


...

Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334


 99%|█████████▉| 497/500 [00:27<00:00, 16.75it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334


100%|██████████| 500/500 [00:27<00:00, 18.19it/s]


Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  47.711973278520034
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334
Dataset Name:  books , AUC Score(benchmark/combined):  55.102774922918805
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.143884892086334


In [34]:

# ==========================================================
# 4. ABLATION: NO NEIGHBOR RECONSTRUCTION
# ==========================================================
print("\n=======================================")
print(f"   ABLATION 3: NO NEIGHBOR RECONSTRUCTION  ")
print("=======================================\n")
for run in range(1):
    train_real_datasets(
        dataset_str=args.dataset, lr=args.lr, epoch_num=args.epoch_num,
        lambda_loss1=0.0, lambda_loss2=args.lambda_loss2, lambda_loss3=args.lambda_loss3, # neighbor=0
        encoder=args.encoder, sample_size=args.sample_size, loss_step=args.loss_step,
        hidden_dim=args.dimension, real_loss=args.real_loss,
        calculate_contextual=args.calculate_contextual, calculate_structural=args.calculate_structural
    )


   ABLATION 3: NO NEIGHBOR RECONSTRUCTION  

Loading books.pt...


  1%|          | 4/500 [00:00<00:29, 16.97it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  43.23997944501542
Dataset Name:  books  Best AUC Score(benchmark/combined):  43.23997944501542
Dataset Name:  books , AUC Score(benchmark/combined):  46.423432682425485
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.423432682425485
Dataset Name:  books , AUC Score(benchmark/combined):  44.57605344295991
Dataset Name:  books  Best AUC Score(benchmark/combined):  46.423432682425485
Dataset Name:  books , AUC Score(benchmark/combined):  48.30935251798561
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561


  1%|          | 6/500 [00:00<00:27, 18.09it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  44.86639260020555
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561
Dataset Name:  books , AUC Score(benchmark/combined):  40.282631038026715
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561
Dataset Name:  books , AUC Score(benchmark/combined):  43.78982528263104
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561
Dataset Name:  books , AUC Score(benchmark/combined):  47.14028776978417
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561
Dataset Name:  books , AUC Score(benchmark/combined):  45.363566289825286
Dataset Name:  books  Best AUC Score(benchmark/combined):  48.30935251798561


...

Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  52.232785200411094
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604


 99%|█████████▉| 497/500 [00:27<00:00, 19.03it/s]

Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  60.41880781089415
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  58.19886947584789
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604


100%|██████████| 500/500 [00:27<00:00, 18.33it/s]


Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.61793422404934
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
Dataset Name:  books , AUC Score(benchmark/combined):  50.0
Dataset Name:  books  Best AUC Score(benchmark/combined):  60.465056526207604
